<a href="https://colab.research.google.com/github/vmodala-10/healing-music-recommender/blob/main/Raw_Data_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("ai4i2020.csv")
print("Initial Shape:", df.shape)
print(df.head())

Initial Shape: (10000, 14)
   UDI Product ID Type  Air temperature [K]  Process temperature [K]  \
0    1     M14860    M                298.1                    308.6   
1    2     L47181    L                298.2                    308.7   
2    3     L47182    L                298.1                    308.5   
3    4     L47183    L                298.2                    308.6   
4    5     L47184    L                298.2                    308.7   

   Rotational speed [rpm]  Torque [Nm]  Tool wear [min]  Machine failure  TWF  \
0                    1551         42.8                0                0    0   
1                    1408         46.3                3                0    0   
2                    1498         49.4                5                0    0   
3                    1433         39.5                7                0    0   
4                    1408         40.0                9                0    0   

   HDF  PWF  OSF  RNF  
0    0    0    0    0  
1    

In [3]:
print("\nINFO:")
print(df.info())
print("\nDESCRIBE:")
print(df.describe())


INFO:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   UDI                      10000 non-null  int64  
 1   Product ID               10000 non-null  object 
 2   Type                     10000 non-null  object 
 3   Air temperature [K]      10000 non-null  float64
 4   Process temperature [K]  10000 non-null  float64
 5   Rotational speed [rpm]   10000 non-null  int64  
 6   Torque [Nm]              10000 non-null  float64
 7   Tool wear [min]          10000 non-null  int64  
 8   Machine failure          10000 non-null  int64  
 9   TWF                      10000 non-null  int64  
 10  HDF                      10000 non-null  int64  
 11  PWF                      10000 non-null  int64  
 12  OSF                      10000 non-null  int64  
 13  RNF                      10000 non-null  int64  
dtypes: float64(3), i

In [4]:
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

print("Columns after cleaning:")
print(df.columns)

Columns after cleaning:
Index(['udi', 'product_id', 'type', 'air_temperature_[k]',
       'process_temperature_[k]', 'rotational_speed_[rpm]', 'torque_[nm]',
       'tool_wear_[min]', 'machine_failure', 'twf', 'hdf', 'pwf', 'osf',
       'rnf'],
      dtype='object')


In [5]:
if 'udi' in df.columns:
    df.drop(columns=['udi'], inplace=True)


In [6]:
print("Missing Values:")
print(df.isnull().sum())
for col in df.columns:
    if df[col].dtype == 'object':
        df[col].fillna(df[col].mode()[0], inplace=True)
    else:
        df[col].fillna(df[col].median(), inplace=True)

Missing Values:
product_id                 0
type                       0
air_temperature_[k]        0
process_temperature_[k]    0
rotational_speed_[rpm]     0
torque_[nm]                0
tool_wear_[min]            0
machine_failure            0
twf                        0
hdf                        0
pwf                        0
osf                        0
rnf                        0
dtype: int64


/tmp/ipykernel_23477/2427256280.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].mode()[0], inplace=True)
/tmp/ipykernel_23477/2427256280.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try

In [7]:
duplicates = df.duplicated().sum()
print("duplicate Rows:", duplicates)

df.drop_duplicates(inplace=True)

duplicate Rows: 0


In [8]:
if 'product_id' in df.columns:
    df['product_id'] = df['product_id'].astype('category')


In [9]:
failure_cols = ['twf', 'hdf', 'pwf', 'osf', 'rnf']

existing_failure_cols = [col for col in failure_cols if col in df.columns]

if 'machine_failure' in df.columns and existing_failure_cols:
    df['calculated_failure'] = df[existing_failure_cols].max(axis=1)

    mismatch = (df['machine_failure'] != df['calculated_failure']).sum()
    print("\nTarget mismatch count:", mismatch)

    df['machine_failure'] = df['calculated_failure']
    df.drop(columns=['calculated_failure'], inplace=True)


Target mismatch count: 27


In [10]:
print("Final Shape:", df.shape)
print("Final Missing Values:")
print(df.isnull().sum())

print("Final Data Types:")
print(df.dtypes)

Final Shape: (10000, 13)
Final Missing Values:
product_id                 0
type                       0
air_temperature_[k]        0
process_temperature_[k]    0
rotational_speed_[rpm]     0
torque_[nm]                0
tool_wear_[min]            0
machine_failure            0
twf                        0
hdf                        0
pwf                        0
osf                        0
rnf                        0
dtype: int64
Final Data Types:
product_id                 category
type                         object
air_temperature_[k]         float64
process_temperature_[k]     float64
rotational_speed_[rpm]        int64
torque_[nm]                 float64
tool_wear_[min]               int64
machine_failure               int64
twf                           int64
hdf                           int64
pwf                           int64
osf                           int64
rnf                           int64
dtype: object


In [11]:
df.to_csv("cleaned_ai4i2020.csv", index=False)
print("Data Cleaned successfully")

Data Cleaned successfully
